In [ ]:
import os
from langchain_community.vectorstores.qdrant import Qdrant
from langchain_qdrant.qdrant import Qdrant
from langchain_openai.embeddings import OpenAIEmbeddings
import qdrant_client
from dotenv import load_dotenv, find_dotenv
 
load_dotenv(find_dotenv())

True

In [2]:
# create a qdrant client
os.environ["QDRANT_HOST"] = "https://8821fb3a-a16e-4069-a1a1-e062fb9c12de.us-east4-0.gcp.cloud.qdrant.io:6333"
os.environ["QDRANT_API_KEY"] = os.getenv("QDRANT_API_KEY")

In [ ]:
# define the client
client = qdrant_client.QdrantClient(
    url=os.environ["QDRANT_HOST"],
    api_key=os.environ["QDRANT_API_KEY"],
)

In [ ]:
# create a collection
os.environ["QDRANT_COLLECTION"] = "my-collection"
client.create_collection(
    collection_name=os.environ["QDRANT_COLLECTION"],
    vectors_config=qdrant_client.http.models.VectorParams(
        size=1536,  # size of the embedding vector for OpenAI
        distance=qdrant_client.http.models.Distance.COSINE,  # distance metric
    ),
)

True

In [28]:
doc_store = Qdrant(
    client=client,
    collection_name=os.environ["QDRANT_COLLECTION"],
    embeddings=OpenAIEmbeddings(),
)

In [17]:
with open("naruto_story.txt", "r", encoding="utf-8") as f:
    naruto_story = f.read()

In [18]:
naruto_story

'Naruto: The Legacy of the Seventh Hokage \n\nThe Hidden Leaf Village was at peace, but Naruto Uzumaki, the Seventh Hokage, felt an unease stirring in his heart. It had been years since the Fourth Great Ninja War, and though the world had healed, shadows of the past never truly disappeared.  \n\nOne evening, as Naruto overlooked the Hokage Monument, a masked figure appeared before him. “Seventh Hokage,” the stranger spoke, his voice carrying the weight of forgotten history. “You have lived in the light for too long. But do you know what lurks beneath?”  \n\nBefore Naruto could respond, the figure vanished, leaving behind a parchment sealed with an ancient Uzumaki clan symbol. Curious and cautious, Naruto took the scroll back to his office. As he unraveled it, his eyes widened in disbelief. It contained a message from the First Hokage, Hashirama Senju.  \n\n*"The true origin of chakra is not what you have been told. Seek the ruins of the Forgotten Temple beyond the Land of Fire. There, 

In [25]:
from langchain_text_splitters import CharacterTextSplitter
# Split the text into chunks
text_splitter = CharacterTextSplitter(
    separator="\n",
	chunk_size=1000,
	chunk_overlap=200,
	length_function=len
)
chunks = text_splitter.split_text(naruto_story)

In [26]:
chunks

['Naruto: The Legacy of the Seventh Hokage \nThe Hidden Leaf Village was at peace, but Naruto Uzumaki, the Seventh Hokage, felt an unease stirring in his heart. It had been years since the Fourth Great Ninja War, and though the world had healed, shadows of the past never truly disappeared.  \nOne evening, as Naruto overlooked the Hokage Monument, a masked figure appeared before him. “Seventh Hokage,” the stranger spoke, his voice carrying the weight of forgotten history. “You have lived in the light for too long. But do you know what lurks beneath?”  \nBefore Naruto could respond, the figure vanished, leaving behind a parchment sealed with an ancient Uzumaki clan symbol. Curious and cautious, Naruto took the scroll back to his office. As he unraveled it, his eyes widened in disbelief. It contained a message from the First Hokage, Hashirama Senju.',
 '*"The true origin of chakra is not what you have been told. Seek the ruins of the Forgotten Temple beyond the Land of Fire. There, you wi

In [ ]:
# add the chunks to the vector store
doc_store.add_texts(chunks)

['9393615768ea492da130e35ddc0ad49d',
 '72cfd8d8392040eeb0e88c6d13b83f8b',
 '38b42f3996104917998839c5d2db8df5']

In [35]:
retriever = doc_store.as_retriever(search_kwargs={"k": 3})

In [33]:
from langchain_google_genai.chat_models import ChatGoogleGenerativeAI

In [42]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
# Define a prompt template
prompt_template = """Answer the following question based on the provided context:

Context: {context}

Question: {query}

Answer: """

prompt = ChatPromptTemplate.from_template(prompt_template)

# Create the LCEL chain
retriever_chain = (
    {"context": retriever, "query": RunnablePassthrough()} 
    | prompt 
    | ChatGoogleGenerativeAI(temperature=0, model="gemini-2.0-flash")
    | StrOutputParser()
)

# Invoke the chain
retriever_chain.invoke("What is the name of the village where Naruto was born?")

'The Hidden Leaf Village'